# Lab 2 - Auditer et corriger un plan de conformité IA

**Objectif.** Détecter des lacunes dans un plan de conformité fictif, relier chaque lacune à une mesure corrective et générer un plan Markdown traçable.

Le contrôle ci-dessous est volontairement simple et fondé sur des règles. Dans un contexte réel, les règles doivent être validées par les équipes juridique, sécurité, données et métier ; elles ne remplacent pas une analyse réglementaire.

## Méthode de travail

Le notebook part d'un plan défaillant, applique des règles de détection transparentes, puis produit un document de correction. Pour chaque résultat, vérifiez trois éléments : le constat correspond-il au texte ? la correction traite-t-elle réellement la cause ? quelle preuve permettra de démontrer l'application du contrôle ? Les tests vérifient le comportement du programme, pas la conformité juridique effective de l'organisation.

In [22]:
from pathlib import Path
from datetime import date
import re

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.txt").exists()), Path.cwd())
ARTIFACTS = PROJECT_ROOT / "notebooks" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

## 1. Définir un plan volontairement incomplet

Chaque élément correspond à une déclaration à auditer. Les exemples reproduisent les thèmes du lab : consentement, tests de biais et journalisation. Les données utilisées sont fictives.

**Pourquoi commencer par ce format ?** Une liste rend les affirmations contrôlables une par une. Dans un véritable audit, elle proviendrait de procédures, contrats, analyses d'impact et preuves techniques. Ici, les trois phrases sont négatives exprès : elles constituent les entrées que le mécanisme doit repérer.

In [23]:
broken_checks = [
    "Les données sont collectées sans consentement de l'utilisateur.",
    "Les tests de biais sont ignorés faute de temps.",
    "Aucune prédiction n'est journalisée.",
]
for item in broken_checks:
    print(f"[!] {item}")

[!] Les données sont collectées sans consentement de l'utilisateur.
[!] Les tests de biais sont ignorés faute de temps.
[!] Aucune prédiction n'est journalisée.


In [24]:
# Test - le scénario couvre les trois familles de lacunes attendues.
assert len(broken_checks) == 3
text = " ".join(broken_checks).lower()
assert all(token in text for token in ("consentement", "biais", "journalisée"))
print("OK - scénario d'audit initialisé.")

OK - scénario d'audit initialisé.


## 2. Détecter les non-conformités avec des règles explicites

Une règle doit être lisible et testable. La fonction renvoie une liste de constats plutôt qu'un simple booléen : cela rend l'audit explicable et facilite la traçabilité.

**Déroulé de la fonction :** `RULES` associe à chaque sujet une expression recherchée, un domaine, une gravité et une correction. `audit_plan` parcourt chaque déclaration puis chaque règle ; si l'expression correspond, elle crée un dictionnaire de constat. La sortie conserve donc le texte d'origine et la recommandation associée. Les expressions régulières sont adaptées à cet exemple ; un audit réel utiliserait un vocabulaire contrôlé et une revue humaine des faux positifs ou négatifs.

In [25]:
RULES = {
    "consentement": {"pattern": r"sans consentement", "domaine": "Juridique", "gravite": "Élevée", "correction": "Obtenir et conserver une preuve de consentement valide avant la collecte."},
    "equite": {"pattern": r"tests? de biais.*ignor", "domaine": "Éthique", "gravite": "Élevée", "correction": "Planifier des métriques d'équité, des seuils et une revue indépendante avant mise en production."},
    "tracabilite": {"pattern": r"aucune prédiction n'est journalisée", "domaine": "Opérationnel", "gravite": "Élevée", "correction": "Journaliser de manière minimisée les versions, décisions et événements utiles à l'audit."},
}

def audit_plan(items, rules=RULES):
    findings = []
    for item in items:
        for rule_id, rule in rules.items():
            if re.search(rule["pattern"], item, flags=re.IGNORECASE):
                findings.append({"regle": rule_id, "constat": item, **{k: v for k, v in rule.items() if k != "pattern"}})
    return findings

findings = audit_plan(broken_checks)
findings

[{'regle': 'consentement',
  'constat': "Les données sont collectées sans consentement de l'utilisateur.",
  'domaine': 'Juridique',
  'gravite': 'Élevée',
  'correction': 'Obtenir et conserver une preuve de consentement valide avant la collecte.'},
 {'regle': 'equite',
  'constat': 'Les tests de biais sont ignorés faute de temps.',
  'domaine': 'Éthique',
  'gravite': 'Élevée',
  'correction': "Planifier des métriques d'équité, des seuils et une revue indépendante avant mise en production."},
 {'regle': 'tracabilite',
  'constat': "Aucune prédiction n'est journalisée.",
  'domaine': 'Opérationnel',
  'gravite': 'Élevée',
  'correction': "Journaliser de manière minimisée les versions, décisions et événements utiles à l'audit."}]

In [26]:
# Test - chaque lacune déclenche exactement un constat, avec une correction.
assert len(findings) == len(broken_checks)
assert {f["regle"] for f in findings} == set(RULES)
assert all(f["correction"] and f["domaine"] for f in findings)
print("OK - toutes les règles attendues ont été détectées.")

OK - toutes les règles attendues ont été détectées.


## 3. Générer le plan corrigé

Le document associe le constat, son domaine, sa gravité et le contrôle recommandé. Sa date est générée automatiquement afin d'éviter qu'une date d'exemple soit prise pour une information réelle.

**Étapes de production :** `render_plan` transforme la liste de constats en texte Markdown ; chaque action reçoit un titre et des champs structurés ; `write_text` écrit le résultat en UTF-8 afin de préserver les accents ; `read_text` l'affiche pour une relecture immédiate. La cellule de test ouvre ensuite le fichier réellement écrit et vérifie son en-tête, les constats et les corrections - elle ne se limite donc pas à tester l'objet Python en mémoire.

In [27]:
def render_plan(findings, owner="Équipe conformité IA"):
    lines = ["# Plan de conformité IA révisé", "", "## Métadonnées", f"- **Responsable :** {owner}", f"- **Date de génération :** {date.today().isoformat()}", "- **Statut :** À valider par les fonctions compétentes", ""]
    for number, finding in enumerate(findings, start=1):
        lines += [f"## Action {number} - {finding['regle'].title()}", f"- **Constat :** {finding['constat']}", f"- **Domaine :** {finding['domaine']}", f"- **Gravité :** {finding['gravite']}", f"- **Contrôle recommandé :** {finding['correction']}", "- **Preuve attendue :** procédure approuvée, résultat de contrôle et responsable désigné.", ""]
    return "\n".join(lines)

plan = render_plan(findings)
output = ARTIFACTS / "lab2_plan_conformite_revise.md"
output.write_text(plan, encoding="utf-8")
print(output.read_text(encoding="utf-8"))

# Plan de conformité IA révisé

## Métadonnées
- **Responsable :** Équipe conformité IA
- **Date de génération :** 2026-11-02
- **Statut :** À valider par les fonctions compétentes

## Action 1 - Consentement
- **Constat :** Les données sont collectées sans consentement de l'utilisateur.
- **Domaine :** Juridique
- **Gravité :** Élevée
- **Contrôle recommandé :** Obtenir et conserver une preuve de consentement valide avant la collecte.
- **Preuve attendue :** procédure approuvée, résultat de contrôle et responsable désigné.

## Action 2 - Equite
- **Constat :** Les tests de biais sont ignorés faute de temps.
- **Domaine :** Éthique
- **Gravité :** Élevée
- **Contrôle recommandé :** Planifier des métriques d'équité, des seuils et une revue indépendante avant mise en production.
- **Preuve attendue :** procédure approuvée, résultat de contrôle et responsable désigné.

## Action 3 - Tracabilite
- **Constat :** Aucune prédiction n'est journalisée.
- **Domaine :** Opérationnel
- **Gravité :

In [28]:
# Test - l'artefact est créé, non vide, et contient chaque constat et correction.
assert output.exists() and output.stat().st_size > 300
saved = output.read_text(encoding="utf-8")
assert saved.startswith("# Plan de conformité IA révisé")
for finding in findings:
    assert finding["constat"] in saved
    assert finding["correction"] in saved
print("OK - plan Markdown complet et traçable.")

OK - plan Markdown complet et traçable.


## 4. Limites et suite de l'audit

Les tests vérifient que la mécanique de détection et le document sont cohérents. Ils ne démontrent pas que les contrôles sont appliqués en pratique. La suite consiste à attribuer un échéancier, collecter les preuves, contrôler les accès aux journaux et faire valider le plan par les responsables concernés.